In [1]:
%run 00_functions.ipynb

c:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data Splitting

In [2]:
# Load the feature-engineered dataset
df = pd.read_csv("../data/lstm/processed/bitcoin_feature_engineered.csv")

# Display the dataset information
print("Dataset shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())

df.head()

Dataset shape: (3189, 9)
Date range: 2018-01-20 to 2026-10-13


,date,open,high,low,close,volume,RSI,SMA,MACD
0,2018-01-20,11579.240234,13082.238281,11567.291016,12858.858398,143900,66.179585,14051.718457,-569.638525
1,2018-01-21,12857.081055,12860.202148,11164.538086,11594.896484,206959,59.335267,13959.219287,-630.799598
2,2018-01-22,11597.452148,11970.898438,10136.767578,10864.833984,201726,55.748913,13764.754492,-729.767823
3,2018-01-23,10869.032227,11437.955078,10056.577148,10903.190430,188365,55.899728,13552.083008,-795.930814
4,2018-01-24,10899.393555,11549.520508,10584.529297,11452.302734,136064,58.101280,13365.694141,-794.893648


In [3]:
# Convert date to datetime format
df["date"] = pd.to_datetime(df["date"])

# Split development and test data chronologically
today = pd.Timestamp.today().normalize()

development_df = df[
    (df["date"] >= "2018-01-01") &
    (df["date"] < "2025-01-01")
].copy()

test_df = df[
    (df["date"] >= "2025-01-01") &
    (df["date"] < today)
].copy()


# Check the split results
print("Development data:")
print("Shape:", development_df.shape)
print("Date range:", development_df["date"].min(), "to", development_df["date"].max())

print("\nTest data:")
print("Shape:", test_df.shape)
print("Date range:", test_df["date"].min(), "to", test_df["date"].max())

Development data:
Shape: (2538, 9)
Date range: 2018-01-20 00:00:00 to 2024-12-31 00:00:00

Test data:
Shape: (650, 9)
Date range: 2025-01-01 00:00:00 to 2026-10-12 00:00:00


In [4]:
# Split development data chronologically into training and validation
split_index = int(len(development_df) * 0.8)

train_df = development_df.iloc[:split_index].copy()
validation_df = development_df.iloc[split_index:].copy()

# Check the split results
print("Training data:")
print("Shape:", train_df.shape)
print("Date range:", train_df["date"].min(), "to", train_df["date"].max())

print("\nValidation data:")
print("Shape:", validation_df.shape)
print("Date range:", validation_df["date"].min(), "to", validation_df["date"].max())

Training data:
Shape: (2030, 9)
Date range: 2018-01-20 00:00:00 to 2023-08-11 00:00:00

Validation data:
Shape: (508, 9)
Date range: 2023-08-12 00:00:00 to 2024-12-31 00:00:00


## 2. Data Normalisation

In [5]:
# Define input features
feature_columns = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "RSI",
    "SMA",
    "MACD"
]

# Define target variable
target_column = "close"

print("Input features:", feature_columns)
print("Target variable:", target_column)

Input features: ['open', 'high', 'low', 'close', 'volume', 'RSI', 'SMA', 'MACD']
Target variable: close


In [6]:
# Initialise scalers
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

# Fit scalers using training data only
feature_scaler.fit(train_df[feature_columns])
target_scaler.fit(train_df[[target_column]])

print("Feature scaler fitted on training data.")
print("Target scaler fitted on training closing prices.")

Feature scaler fitted on training data.
Target scaler fitted on training closing prices.


In [7]:
# Transform input features
X_train_scaled = feature_scaler.transform(train_df[feature_columns])
X_val_scaled = feature_scaler.transform(validation_df[feature_columns])

# Transform target variable
y_train_scaled = target_scaler.transform(train_df[[target_column]])
y_val_scaled = target_scaler.transform(validation_df[[target_column]])

# Check the shapes
print("Training data:", X_train_scaled.shape, y_train_scaled.shape)
print("Validation data:", X_val_scaled.shape, y_val_scaled.shape)

Training data: (2030, 8) (2030, 1)
Validation data: (508, 8) (508, 1)


In [8]:
# Create time-series sequences for LSTM
def create_sequences(X, y, look_back):
    X_sequences = []
    y_targets = []

    for i in range(look_back, len(X)):
        X_sequences.append(X[i - look_back:i])
        y_targets.append(y[i])

    return np.array(X_sequences), np.array(y_targets)

In [9]:
# Define the look-back window
look_back = 90

# Create sequences for each dataset
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled, y_train_scaled, look_back
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled, y_val_scaled, look_back
)

# Check sequence shapes
print("Training sequences:", X_train_seq.shape, y_train_seq.shape)
print("Validation sequences:", X_val_seq.shape, y_val_seq.shape)

Training sequences: (1940, 90, 8) (1940, 1)
Validation sequences: (418, 90, 8) (418, 1)


## 3. LSTM Model Development

In [10]:
model = Sequential([
    LSTM(64, input_shape=(look_back, len(feature_columns))),
    Dropout(0.2),
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

c:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        18,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,753 (73.25 KB)

 Trainable params: 18,753 (73.25 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping],
    shuffle=False
)

Epoch 1/50
61/61 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.0164 - mae: 0.0896 - val_loss: 0.1777 - val_mae: 0.3923
Epoch 2/50
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0143 - mae: 0.0924 - val_loss: 0.1194 - val_mae: 0.3205
Epoch 3/50
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0115 - mae: 0.0778 - val_loss: 0.0549 - val_mae: 0.2126
Epoch 4/50
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0078 - mae: 0.0636 - val_loss: 0.0293 - val_mae: 0.1523
Epoch 5/50
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0051 - mae: 0.0513 - val_loss: 0.0137 - val_mae: 0.0996
Epoch 6/50
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0035 - mae: 0.0422 - val_loss: 0.0131 - val_mae: 0.0994
Epoch 7/50
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0029 - mae: 0.0369 - val_loss: 0.0107 - val_mae: 0.0891
Epoch 8/50
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0025 - mae: 0.0337 - val_loss: 0.0130 - val_mae: 0.1002
Epoch 9/50
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.002

In [12]:
best_epoch = history.epoch[
    history.history["val_loss"].index(
        min(history.history["val_loss"])
    )
] + 1

best_val_loss = min(history.history["val_loss"])
best_val_mae = history.history["val_mae"][
    best_epoch - 1
]

print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)
print("Best validation MAE:", best_val_mae)

Best epoch: 7
Best validation loss: 0.010701630264520645
Best validation MAE: 0.08909226953983307


In [13]:
# Predict validation closing prices
y_val_pred_scaled = model.predict(X_val_seq)

# Convert predictions and actual values back to BTC prices
y_val_pred = target_scaler.inverse_transform(y_val_pred_scaled)
y_val_actual = target_scaler.inverse_transform(y_val_seq)

# Calculate evaluation metrics
val_mae = mean_absolute_error(y_val_actual, y_val_pred)
val_rmse = np.sqrt(mean_squared_error(y_val_actual, y_val_pred))
val_r2 = r2_score(y_val_actual, y_val_pred)

print(f"Validation MAE: ${val_mae:,.2f}")
print(f"Validation RMSE: ${val_rmse:,.2f}")
print(f"Validation R²: {val_r2:.4f}")

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
Validation MAE: $5,728.56
Validation RMSE: $6,651.66
Validation R²: 0.8303


In [14]:
# 1. Check the alignment and number of validation predictions
print("Validation sequence count:", len(y_val_actual))
print("Validation prediction count:", len(y_val_pred))
print("Validation dataframe rows:", len(validation_df))

# 2. Calculate actual daily price direction
actual_change = (
    y_val_actual[1:, 0] - y_val_actual[:-1, 0]
)

# 3. Calculate predicted direction
predicted_change = (
    y_val_pred[1:, 0] - y_val_actual[:-1, 0]
)

# 4. Calculate directional accuracy
actual_direction = np.sign(actual_change)
predicted_direction = np.sign(predicted_change)

direction_accuracy = np.mean(
    actual_direction == predicted_direction
) * 100

print(f"\nLSTM Directional Accuracy: {direction_accuracy:.2f}%")
print("Direction comparison count:", len(actual_direction))

Validation sequence count: 418
Validation prediction count: 418
Validation dataframe rows: 508

LSTM Directional Accuracy: 47.72%
Direction comparison count: 417


In [15]:
# Baseline: tomorrow's closing price equals today's closing price

# Actual price movement from one day to the next
actual_change = (
    y_val_actual[1:, 0] - y_val_actual[:-1, 0]
)

# Baseline predicts no price movement
baseline_change = np.zeros_like(actual_change)

# Compare actual and baseline-predicted directions
actual_direction = np.sign(actual_change)
baseline_direction = np.sign(baseline_change)

baseline_accuracy = np.mean(
    actual_direction == baseline_direction
) * 100

# Count actual price movements
up_days = np.sum(actual_change > 0)
down_days = np.sum(actual_change < 0)
unchanged_days = np.sum(actual_change == 0)

print(f"Baseline Directional Accuracy: {baseline_accuracy:.2f}%")
print(f"Actual Up Days: {up_days}")
print(f"Actual Down Days: {down_days}")
print(f"Unchanged Days: {unchanged_days}")
print("Direction comparison count:", len(actual_change))

Baseline Directional Accuracy: 0.00%
Actual Up Days: 220
Actual Down Days: 197
Unchanged Days: 0
Direction comparison count: 417


## 4. Final Model Training

In [16]:
# Combine training and validation data
development_df = pd.concat(
    [train_df, validation_df],
    ignore_index=True
)

print("Development data shape:", development_df.shape)
print("Start date:", development_df["date"].min())
print("End date:", development_df["date"].max())

Development data shape: (2538, 9)
Start date: 2018-01-20 00:00:00
End date: 2024-12-31 00:00:00


In [17]:
# Prepare the full historical dataset for final model training
today = pd.Timestamp.today().normalize()

final_train_df = df[df["date"] < today].copy()

# Refit scalers using all available data before today
feature_scaler_final = MinMaxScaler()
target_scaler_final = MinMaxScaler()

feature_scaler_final.fit(
    final_train_df[feature_columns]
)

target_scaler_final.fit(
    final_train_df[[target_column]]
)

print("Final training data shape:", final_train_df.shape)
print(
    "Final training date range:",
    final_train_df["date"].min(),
    "to",
    final_train_df["date"].max()
)
print("Feature scaler fitted on final training data.")
print("Target scaler fitted on final training data.")

Final training data shape: (3188, 9)
Final training date range: 2018-01-20 00:00:00 to 2026-10-12 00:00:00
Feature scaler fitted on final training data.
Target scaler fitted on final training data.


In [18]:

# Transform final training data using the final scalers
X_final_scaled = feature_scaler_final.transform(
    final_train_df[feature_columns]
)

y_final_scaled = target_scaler_final.transform(
    final_train_df[[target_column]]
)

print("X final shape:", X_final_scaled.shape)
print("y final shape:", y_final_scaled.shape)

X final shape: (3188, 8)
y final shape: (3188, 1)


In [19]:
X_final_seq, y_final_seq = create_sequences(
    X_final_scaled,
    y_final_scaled,
    look_back
)

print("X final sequences:", X_final_seq.shape)
print("y final sequences:", y_final_seq.shape)

X final sequences: (3098, 90, 8)
y final sequences: (3098, 1)


In [20]:
final_model = Sequential([
    LSTM(64, input_shape=(look_back, len(feature_columns))),
    Dropout(0.2),
    Dense(1)
])

final_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

final_model.summary()

c:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        18,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,753 (73.25 KB)

 Trainable params: 18,753 (73.25 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
final_history = final_model.fit(
    X_final_seq,
    y_final_seq,
    epochs=45,
    batch_size=32,
    shuffle=False
)

Epoch 1/45
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0085 - mae: 0.0706
Epoch 2/45
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0087 - mae: 0.0754
Epoch 3/45
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0039 - mae: 0.0500
Epoch 4/45
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0018 - mae: 0.0304
Epoch 5/45
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0016 - mae: 0.0276
Epoch 6/45
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0015 - mae: 0.0278
Epoch 7/45
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0016 - mae: 0.0276
Epoch 8/45
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0015 - mae: 0.0260
Epoch 9/45
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0015 - mae: 0.0268
Epoch 10/45
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0017 - mae: 0.0281
Epoch 11/45
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0016 - mae: 0.0278
Epoch 12/45
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0015 - mae: 0.0269
Epoch 13/45
97/97 ━━━━━━━━━━━━━━━━━━━

In [22]:
# Include the 90 days before the test period as historical context
test_context_df = pd.concat(
    [development_df.tail(look_back), test_df],
    ignore_index=True
)

# Transform using scalers fitted on training data only
X_test_context_scaled = feature_scaler.transform(
    test_context_df[feature_columns]
)

y_test_context_scaled = target_scaler.transform(
    test_context_df[[target_column]]
)

print("Test context shape:", test_context_df.shape)
print("Test data shape:", test_df.shape)

Test context shape: (740, 9)
Test data shape: (650, 9)


In [23]:
# Create test sequences with historical context
X_test_seq, y_test_seq = create_sequences(
    X_test_context_scaled,
    y_test_context_scaled,
    look_back
)

print("X test sequences:", X_test_seq.shape)
print("y test sequences:", y_test_seq.shape)
print("Expected test rows:", len(test_df))

X test sequences: (650, 90, 8)
y test sequences: (650, 1)
Expected test rows: 650


In [24]:
# Predict test closing prices using the historical model
y_test_pred_scaled = model.predict(
    X_test_seq,
    verbose=0
)

# Convert predictions back to actual BTC prices
y_test_pred = target_scaler.inverse_transform(
    y_test_pred_scaled
)

y_test_actual = target_scaler.inverse_transform(
    y_test_seq
)

print("Prediction shape:", y_test_pred.shape)
print("Actual price shape:", y_test_actual.shape)

print("\nFirst 5 predicted closing prices:")
print(y_test_pred[:5].flatten())

print("\nFirst 5 actual closing prices:")
print(y_test_actual[:5].flatten())

Prediction shape: (650, 1)
Actual price shape: (650, 1)

First 5 predicted closing prices:
[82197.695 82083.93  82515.57  83118.93  83576.59 ]

First 5 actual closing prices:
[94560.1953125 96959.984375  98167.4609375 98219.875     98347.6953125]


In [25]:
# Evaluate historical test predictions
test_mae = mean_absolute_error(
    y_test_actual,
    y_test_pred
)

test_mse = mean_squared_error(
    y_test_actual,
    y_test_pred
)

test_rmse = np.sqrt(test_mse)

test_r2 = r2_score(
    y_test_actual,
    y_test_pred
)

print(f"Test MAE: ${test_mae:,.2f}")
print(f"Test MSE: {test_mse:,.2f}")
print(f"Test RMSE: ${test_rmse:,.2f}")
print(f"Test R²: {test_r2:.4f}")

Test MAE: $11,875.77
Test MSE: 166,362,238.66
Test RMSE: $12,898.15
Test R²: 0.4587


In [26]:
# Get the latest 90 days of feature data before today
today = pd.Timestamp.today().normalize()

historical_df = df[df["date"] < today].copy()
latest_data = historical_df[feature_columns].tail(look_back)

print("Latest input date range:",
      historical_df["date"].tail(look_back).min(),
      "to",
      historical_df["date"].tail(look_back).max())

print("Latest input shape:", latest_data.shape)

# Scale the latest data using the final feature scaler
latest_data_scaled = feature_scaler_final.transform(
    latest_data
)

# Reshape data for LSTM: (samples, timesteps, features)
latest_sequence = latest_data_scaled.reshape(
    1, look_back, len(feature_columns)
)

# Predict the next closing price
next_close_scaled = final_model.predict(
    latest_sequence,
    verbose=0
)

# Convert prediction back to actual BTC price
next_close = target_scaler_final.inverse_transform(
    next_close_scaled
)[0, 0]

# Get the latest actual closing price
latest_close = historical_df["close"].iloc[-1]

print("\nLatest available date:", historical_df["date"].iloc[-1])
print(f"Latest BTC Close Price: ${latest_close:,.2f}")

print("\nForecast date:", today.date())
print(f"Forecast Close Price: ${next_close:,.2f}")

Latest input date range: 2026-07-15 00:00:00 to 2026-10-12 00:00:00
Latest input shape: (90, 8)

Latest available date: 2026-10-12 00:00:00
Latest BTC Close Price: $81,842.29

Forecast date: 2026-10-13
Forecast Close Price: $83,546.05


## 7. Save Final Model and Scalers

In [27]:
# Create the models directory
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Save the trained LSTM model
final_model.save(models_dir / "lstm_model.keras")

# Save the scalers
joblib.dump(
    feature_scaler_final,
    models_dir / "feature_scaler.pkl"
)

joblib.dump(
    target_scaler_final,
    models_dir / "target_scaler.pkl"
)

print("Final model and scalers saved successfully.")

Final model and scalers saved successfully.
